# 安裝 LLM 伺服器（只做一次）

在 server 電腦用 VS Code 打開，由上而下按 ▶ 執行全部 cell。之後每次用只要開 `start_server.ipynb`。

要先準備好：

- Python，VS Code 裝好 Python 和 Jupyter 擴充功能
- Ollama：在 PowerShell 執行 `irm https://ollama.com/install.ps1 | iex`，裝好後重開 VS Code

## 1. 設定

In [ ]:
import ipaddress
import json
import os
import shutil
import socket
import subprocess
import tempfile
import time
import urllib.request
from pathlib import Path

MODEL = "qwen3:14b"  # 想換模型改這裏；install_server 和 start_server 兩個檔案要一樣
# 剛裝完 Ollama 時 VS Code 可能還沒拿到新的 PATH，所以也找預設的安裝位置
OLLAMA = Path(shutil.which("ollama")
              or Path(os.environ["LOCALAPPDATA"]) / "Programs" / "Ollama" / "ollama.exe")
if not OLLAMA.exists():
    raise RuntimeError("找不到 Ollama：先在 PowerShell 執行 irm https://ollama.com/install.ps1 | iex，"
                       "裝好後重開 VS Code 再試")
URL = "http://127.0.0.1:11434"
RULE = "Ollama LLM Server"
LOG = Path(tempfile.gettempdir()) / "ollama_server.log"


def ollama_up():
    try:
        urllib.request.urlopen(URL, timeout=2)
        return True
    except OSError:
        return False


def wait_up(seconds=60):
    for _ in range(seconds):
        if ollama_up():
            return True
        time.sleep(1)
    return False


def post(path, body, timeout=600):
    req = urllib.request.Request(URL + path, json.dumps(body).encode("utf-8"),
                                 {"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=timeout)


print("設定完成，模型：", MODEL, "，Ollama：", OLLAMA)

## 2. 下載模型（`qwen3:14b` 約 9GB）

In [ ]:
if not ollama_up():
    subprocess.Popen([str(OLLAMA), "serve"], creationflags=subprocess.CREATE_NO_WINDOW,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not wait_up():
        raise RuntimeError("Ollama 60 秒內沒有啟動")

shown = set()
with post("/api/pull", {"model": MODEL}, timeout=None) as r:
    for line in r:
        d = json.loads(line)
        if "error" in d:
            raise RuntimeError(d["error"])
        if d.get("total"):
            # 只在每 10% 印一次，不然會印幾千行
            pct = d.get("completed", 0) * 100 // d["total"] // 10 * 10
            key = (d.get("digest"), pct)
            if d["total"] > 100_000_000 and key not in shown:
                shown.add(key)
                print(f"下載中… {pct}%")
        elif d.get("status") == "success":
            print(f"{MODEL} 下載完成")

## 3. 開防火牆

會彈出「是否允許變更」的視窗，按 **是**。

只讓同一個網絡和 Tailscale（`100.64.0.0/10`）的電腦連入。不分私人／公用網絡，因為 Windows 新連的 Wi-Fi 預設是公用。指定了 Ollama 程式，Windows 就不會彈出自己的防火牆視窗（那個視窗按錯會封鎖 Ollama）。

In [ ]:
# 防火牆要管理員權限：把指令寫進暫存檔，再用管理員身份執行它
script = Path(tempfile.gettempdir()) / "ollama_firewall.cmd"
script.write_text(
    f'netsh advfirewall firewall delete rule name="{RULE}"\r\n'
    f'netsh advfirewall firewall add rule name="{RULE}" dir=in action=allow program="{OLLAMA}" '
    f'protocol=TCP localport=11434 remoteip=localsubnet,100.64.0.0/10 profile=any\r\n',
    encoding="ascii")
subprocess.run(["powershell", "-NoProfile", "-Command",
                "Start-Process -FilePath $env:FW_SCRIPT -Verb RunAs -Wait -WindowStyle Hidden"],
               env={**os.environ, "FW_SCRIPT": str(script)}, check=True)

check = subprocess.run(["netsh", "advfirewall", "firewall", "show", "rule", f"name={RULE}"],
                       capture_output=True)
print("防火牆已開放" if check.returncode == 0 else "防火牆規則沒有加到，是不是按了「否」？再執行一次")